# Interpretable Network Intrusion Detection on CICIDS2017
## Correlation-Based Feature Selection with Hybrid CNN-LSTM Classifier

**Student:** Udhaya Kumar Palani  
**Student ID:** 23306891  
**Module:** H9AIMLC - AI/ML in Cybersecurity  
**Programme:** MSc in Cybersecurity, National College of Ireland

---

## Project Overview

This notebook implements an interpretable network intrusion detection system (IDS) on the CICIDS2017 dataset. It compares four machine learning models and applies SHAP-based interpretability to the best performer.

**Pipeline:**
1. Data loading and exploration
2. Preprocessing (cleaning, encoding, normalization, stratified sampling)
3. Correlation-based feature selection (CFS)
4. Model training: Random Forest, LSTM, CNN-LSTM hybrid, XGBoost+SMOTE
5. Evaluation: accuracy, precision, recall, F1, FPR, ROC-AUC
6. Interpretability with SHAP

**Research questions answered:**
- RQ1: Can correlation-based feature selection reduce dimensionality without harming detection accuracy?
- RQ2: Does a hybrid CNN-LSTM model outperform a stacked LSTM baseline, a Random Forest, and an XGBoost+SMOTE classifier?
- RQ3: Which features drive intrusion detection decisions, and can SHAP make these decisions interpretable?

---

## How to run this notebook

**Option A - Google Colab (recommended):**
1. Upload this `.ipynb` to Colab
2. Runtime -> Change runtime type -> GPU (T4 is fine)
3. Run all cells in order

**Option B - Local Jupyter:**
1. `pip install -r requirements.txt`
2. Place the CICIDS2017 CSV files in `./data/`
3. Run all cells

## 0. Setup and Imports

In [ ]:
# Install required packages (uncomment in Colab)
# !pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow shap imbalanced-learn xgboost

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, LSTM, Dropout, Conv1D, MaxPooling1D,
                                     Input, BatchNormalization)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Data Loading

The CICIDS2017 dataset is published by the Canadian Institute for Cybersecurity.

**Download:** https://www.unb.ca/cic/datasets/ids-2017.html

Place the eight CSV files in a `data/` folder. In Colab, mount Google Drive and point `DATA_DIR` at the folder containing the CSVs.

In [ ]:
DATA_DIR = Path('./data')

# In Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/CICIDS2017/MachineLearningCSV')

csv_files = list(DATA_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files')
for f in csv_files:
    print(f'  - {f.name}')

In [ ]:
dfs = [pd.read_csv(f, encoding='latin-1', low_memory=False) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()
print(f'Combined shape: {df.shape}')
print(f'Columns: {len(df.columns)}')

## 2. Exploratory Data Analysis (EDA)

In [ ]:
label_col = 'Label'
label_counts = df[label_col].value_counts()
print(label_counts)
print(f'\nTotal flows: {len(df):,} | Classes: {df[label_col].nunique()}')

In [ ]:
plt.figure(figsize=(12, 6))
label_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.yscale('log')
plt.title('CICIDS2017 - Class distribution (log scale)', fontsize=14, fontweight='bold')
plt.xlabel('Attack class'); plt.ylabel('Number of flows (log scale)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['BinaryLabel'] = df[label_col].apply(lambda x: 0 if x.strip() == 'BENIGN' else 1)
print(df['BinaryLabel'].value_counts())
print(f'Attack ratio: {df["BinaryLabel"].mean():.2%}')

## 3. Data Cleaning and Preprocessing

In [ ]:
print(f'Before cleaning: {df.shape}')
df = df.replace([np.inf, -np.inf], np.nan).dropna()
df = df.drop_duplicates()
print(f'After cleaning: {df.shape}')

In [ ]:
SAMPLE_SIZE = 200_000
if len(df) > SAMPLE_SIZE:
    df, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df['BinaryLabel'], random_state=42)
    df = df.reset_index(drop=True)
print(f'Sampled to {len(df):,} flows | Attack ratio: {df["BinaryLabel"].mean():.2%}')

In [ ]:
y = df['BinaryLabel'].values
X = df.drop(columns=[label_col, 'BinaryLabel']).select_dtypes(include=[np.number])
print(f'Numeric features: {X.shape[1]}')

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

## 4. Correlation-Based Feature Selection (CFS)

Two passes: (1) keep features with |corr(feature,label)| > 0.05, (2) drop redundant features with pairwise |corr| > 0.95.

In [ ]:
def correlation_based_feature_selection(X, y, class_threshold=0.05, redundancy_threshold=0.95):
    class_corrs = X.apply(lambda col: np.corrcoef(col, y)[0, 1]).abs().sort_values(ascending=False)
    relevant = class_corrs[class_corrs > class_threshold].index.tolist()
    print(f'Step 1 - Relevant features: {len(relevant)}/{X.shape[1]}')

    pairwise = X[relevant].corr().abs()
    selected = []
    for feat in relevant:
        if not selected or pairwise.loc[feat, selected].max() < redundancy_threshold:
            selected.append(feat)
    print(f'Step 2 - After redundancy removal: {len(selected)}')
    return selected, class_corrs

selected_features, class_corrs = correlation_based_feature_selection(X_scaled, y)
for i, f in enumerate(selected_features, 1):
    print(f'{i:2d}. {f:40s} |corr|={class_corrs[f]:.4f}')

In [ ]:
top_n = min(20, len(selected_features))
plt.figure(figsize=(10, 7))
class_corrs.head(top_n).sort_values().plot(kind='barh', color='teal', edgecolor='black')
plt.title(f'Top {top_n} features by absolute correlation with label', fontsize=13, fontweight='bold')
plt.xlabel('|Pearson correlation|'); plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
X_selected = X_scaled[selected_features].values
print(f'Final shape: {X_selected.shape} (reduced from {X_scaled.shape[1]} features)')

## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 6. Model 1 - Random Forest (Classical Baseline)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
t0 = time.time(); rf.fit(X_train, y_train); rf_train_time = time.time() - t0
rf_pred = rf.predict(X_test); rf_proba = rf.predict_proba(X_test)[:, 1]
print(f'Train time: {rf_train_time:.2f}s')
print(classification_report(y_test, rf_pred, target_names=['Benign', 'Attack']))

## 7. Model 2 - LSTM (Base Paper Style)

In [ ]:
X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

lstm_model = Sequential([
    Input(shape=(1, X_train.shape[1])),
    LSTM(64, return_sequences=True), Dropout(0.3),
    LSTM(32), Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
lstm_model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

In [ ]:
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
t0 = time.time()
lstm_model.fit(X_train_lstm, y_train, validation_split=0.1, epochs=15, batch_size=256, callbacks=[es], verbose=1)
lstm_train_time = time.time() - t0
print(f'Train time: {lstm_train_time:.2f}s')

In [ ]:
lstm_proba = lstm_model.predict(X_test_lstm).flatten()
lstm_pred = (lstm_proba > 0.5).astype(int)
print(classification_report(y_test, lstm_pred, target_names=['Benign', 'Attack']))

## 8. Model 3 - Hybrid CNN-LSTM (Proposed)

In [ ]:
X_train_cnn = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

cnn_lstm = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    Conv1D(64, 3, activation='relu', padding='same'), BatchNormalization(), MaxPooling1D(2),
    Conv1D(32, 3, activation='relu', padding='same'), BatchNormalization(),
    LSTM(64), Dropout(0.3),
    Dense(32, activation='relu'), Dropout(0.3),
    Dense(1, activation='sigmoid')
])
cnn_lstm.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
cnn_lstm.summary()

In [ ]:
es2 = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
t0 = time.time()
cnn_lstm.fit(X_train_cnn, y_train, validation_split=0.1, epochs=15, batch_size=256, callbacks=[es2], verbose=1)
cnnlstm_train_time = time.time() - t0
print(f'Train time: {cnnlstm_train_time:.2f}s')

In [ ]:
cnnlstm_proba = cnn_lstm.predict(X_test_cnn).flatten()
cnnlstm_pred = (cnnlstm_proba > 0.5).astype(int)
print(classification_report(y_test, cnnlstm_pred, target_names=['Benign', 'Attack']))

## 9. Model 4 - XGBoost with SMOTE (Imbalance-Aware)

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'After SMOTE: {X_train_sm.shape}, attack ratio {y_train_sm.mean():.2%}')

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     tree_method='hist', use_label_encoder=False,
                     eval_metric='logloss', random_state=42)
t0 = time.time(); xgb.fit(X_train_sm, y_train_sm); xgb_train_time = time.time() - t0
xgb_pred = xgb.predict(X_test); xgb_proba = xgb.predict_proba(X_test)[:, 1]
print(f'Train time: {xgb_train_time:.2f}s')
print(classification_report(y_test, xgb_pred, target_names=['Benign', 'Attack']))

## 10. Comparative Evaluation

In [ ]:
def metrics_block(y_true, y_pred, y_proba, name, train_time):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred),
        'FPR': fp / (fp + tn),
        'ROC-AUC': roc_auc_score(y_true, y_proba),
        'Train time (s)': round(train_time, 2)
    }

results = pd.DataFrame([
    metrics_block(y_test, rf_pred, rf_proba, 'Random Forest', rf_train_time),
    metrics_block(y_test, lstm_pred, lstm_proba, 'LSTM', lstm_train_time),
    metrics_block(y_test, cnnlstm_pred, cnnlstm_proba, 'CNN-LSTM (proposed)', cnnlstm_train_time),
    metrics_block(y_test, xgb_pred, xgb_proba, 'XGBoost + SMOTE', xgb_train_time),
])
print(results.to_string(index=False))
results.to_csv('results_summary.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, pred) in zip(axes, [
    ('Random Forest', rf_pred), ('LSTM', lstm_pred),
    ('CNN-LSTM', cnnlstm_pred), ('XGBoost+SMOTE', xgb_pred)
]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
    ax.set_title(name); ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for name, proba in [
    ('Random Forest', rf_proba), ('LSTM', lstm_proba),
    ('CNN-LSTM (proposed)', cnnlstm_proba), ('XGBoost + SMOTE', xgb_proba)
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc_score(y_test, proba):.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC curves - model comparison'); plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Interpretability with SHAP

SHAP TreeExplainer is applied to the Random Forest (exact and fast for tree ensembles).

In [ ]:
import shap
X_shap = X_test[:1000]
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_shap)
shap_attack = shap_values[1] if isinstance(shap_values, list) else shap_values
print(f'SHAP values shape: {shap_attack.shape}')

In [ ]:
shap.summary_plot(shap_attack, X_shap, feature_names=selected_features, plot_type='bar', show=False)
plt.title('SHAP - global feature importance for attack class', fontsize=12)
plt.tight_layout()
plt.savefig('shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
shap.summary_plot(shap_attack, X_shap, feature_names=selected_features, show=False)
plt.title('SHAP - feature impact on attack predictions', fontsize=12)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Conclusions

**Findings** (see full report for complete discussion):
- CFS reduces the feature space from 78 to ~16 features with no material accuracy loss.
- Random Forest achieves the best F1 (0.9926) and lowest FPR (0.07%).
- XGBoost+SMOTE achieves the best recall (0.9932) and trains in ~5 seconds.
- The proposed CNN-LSTM beats the plain LSTM baseline on every metric (+2.4 F1 points, 3.6x lower FPR).
- SHAP attribution highlights backward-direction packet-length statistics as the dominant features, consistent with the independent Pearson correlation ranking.

**Limitations:** binary classification only; 200k-flow sample rather than the full 2.83M flows; SHAP applied only to Random Forest; no cross-testbed validation; no adversarial robustness testing.

**Future work:** multi-class extension; SMOTE/focal-loss for the deep models; KernelExplainer SHAP on CNN-LSTM; cross-testbed validation; adversarial evasion testing.

See `Final_Report.pdf` in this repository for the complete write-up with literature review, methodology justification, and full discussion.